# أنماط العودية المتقدمة في UnifyWeaver

يوضح هذا الدفتر من دفاتر الملاحظات أنماط العودية الأربعة الرئيسية التي يمكن لـ UnifyWeaver اكتشافها وتحسينها:

1. **عودية الذيل (Tail Recursion)** - حلقات تكرارية باستخدام مجمعات القيمة (accumulators)
2. **العودية الخطية (Linear Recursion)** - استدعاء عودي واحد مع الحفظ في الذاكرة (memoization)
3. **العودية الشجرية (Tree Recursion)** - استدعاءات عودية متعددة على أجزاء الهيكل
4. **العودية المتبادلة (Mutual Recursion)** - محددات تستدعي بعضها البعض في دورات

## الأهداف التعليمية

- فهم أنماط العودية المختلفة
- معرفة كيف يكتشف UnifyWeaver كل نمط ويحسنه
- مقارنة خصائص الأداء
- معرفة متى تستخدم كل نمط

## الإعداد

تهيئة بيئة UnifyWeaver.

In [ ]:
% تحميل التهيئة
['../init'].

% تحميل الوحدات النمطية اللازمة
use_module(unifyweaver(core/recursive_compiler)).
use_module(unifyweaver(core/advanced/pattern_matchers)).

## النمط 1: عودية الذيل (Tail Recursion)

تستخدم عودية الذيل مجمعًا (accumulator) لنقل النتائج الوسيطة، ويكون الاستدعاء العودي هو **الإجراء الأخير** في الدالة.

### مثال: عد العناصر في قائمة

In [ ]:
% تعريف count_items بعودية الذيل
:- dynamic count_items/3.

% الحالة الأساسية: قائمة فارغة، إرجاع المجمع
count_items([], Acc, Acc).

% الحالة العودية: زيادة المجمع، وتطبيق العودية على الذيل
count_items([_|T], Acc, N) :-
    Acc1 is Acc + 1,
    count_items(T, Acc1, N).  % ← موضع الذيل!

### الاختبار في Prolog

In [ ]:
% اختبار: عد العناصر في [a,b,c,d,e]
\+ \+ (
    count_items([a,b,c,d,e], 0, _N),
    format('Count: ~w~n', [_N])
).

### التحقق من اكتشاف النمط

In [ ]:
% التحقق مما إذا تم اكتشافها كعودية ذيل
\+ \+ (
    is_tail_recursive_accumulator(count_items/3, _AccInfo),
    format('Tail recursive: ~w~n', [_AccInfo])
).

### التجميع إلى Bash

In [ ]:
% تجميع وحفظ
\+ \+ (
    compile_recursive(count_items/3, [], _BashCode),
    setup_call_cleanup(
        open('../output/count_items_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled count_items to Bash with tail recursion optimization')
).

### اختبار كود Bash المولد

In [ ]:
%%bash
source ../output/count_items_demo.sh
echo "عد العناصر في [a,b,c,d,e]:"
count_items "[a,b,c,d,e]" 0 ""

## النمط 2: العودية الخطية (Linear Recursion)

تحتوي العودية الخطية على استدعاء عودي **واحد فقط** لكل بند، مع حدوث العمليات الحسابية بعد عودة الاستدعاء العودي.

### مثال: العاملي (Factorial)

In [ ]:
% تعريف المضروب (factorial)
:- dynamic factorial/2.

% الحالة الأساسية
factorial(0, 1).

% الحالة العودية: استدعاء عودي واحد فقط
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),  % ← استدعاء عودي واحد
    F is N * F1.        % ← العمليات الحسابية بعد الاستدعاء

### الاختبار في Prolog

In [ ]:
% اختبار: مضروب 5
\+ \+ (
    factorial(5, _F),
    format('5! = ~w~n', [_F])
).

### التحقق من اكتشاف النمط

In [ ]:
% التحقق مما إذا تم اكتشافها كعودية خطية
is_linear_recursive_streamable(factorial/2),
writeln('✓ Detected as linear recursion').

### التجميع إلى Bash

In [ ]:
% تجميع وحفظ
\+ \+ (
    compile_recursive(factorial/2, [], _BashCode),
    % الاحتفاظ بتعريفات الدوال فقط؛ حيث يتعامل Brush مع السكربتات المستوردة عبر source كتنفيذ مباشر
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Auto-execute when run directly (not when sourced)"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/factorial_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled factorial to Bash with fold-based linear recursion')
).

### اختبار كود Bash المولد

In [ ]:
%%bash
source ../output/factorial_demo.sh
echo "مضروب 5:"
factorial 5 ""
echo ""
echo "مضروب 10:"
factorial 10 ""

## النمط 3: العودية الشجرية (Tree Recursion)

تُجري العودية الشجرية استدعاءات عودية **متعددة** لمعالجة أجزاء مختلفة من الهيكل البرمجي.

### مثال: مجموع الشجرة (Tree Sum)

In [ ]:
% تعريف tree_sum للأشجار الثنائية
% تنسيق الشجرة: [القيمة, الشجرة_الفرعية_اليسرى, الشجرة_الفرعية_اليمنى] أو []
:- dynamic tree_sum/2.

% الحالة الأساسية: الشجرة الفارغة مجموعها 0
tree_sum([], 0).

% الحالة العودية: المجموع = القيمة + مجموع_اليسار + مجموع_اليمين
tree_sum([V, L, R], Sum) :-
    tree_sum(L, LS),   % ← الاستدعاء العودي الأول
    tree_sum(R, RS),   % ← الاستدعاء العودي الثاني
    Sum is V + LS + RS.

### الاختبار في Prolog

In [ ]:
% اختبار: tree_sum لـ [5, [3, [1, [], []], []], [2, [], []]]
%       5
%      / \
%     3   2
%    /
%   1
\+ \+ (
    tree_sum([5, [3, [1, [], []], []], [2, [], []]], _Sum),
    format('Tree sum: ~w (expected 11)~n', [_Sum])
).

### التجميع إلى Bash

In [ ]:
% تجميع وحفظ
\+ \+ (
    compile_recursive(tree_sum/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/tree_sum_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled tree_sum to Bash with tree recursion')
).

### اختبار كود Bash المولد

In [ ]:
%%bash
source ../output/tree_sum_demo.sh
echo "مجموع الشجرة لـ [5,[3,[1,[],[]],[]],[2,[],[]]]:"
tree_sum "[5,[3,[1,[],[]],[]],[2,[],[]]]"

## النمط 4: العودية المتبادلة (Mutual Recursion)

تحدث العودية المتبادلة عندما يستدعي محددان أو أكثر بعضهما البعض في دورة متكررة.

### مثال: الزوجي (Even) والفردي (Odd)

In [ ]:
% تعريف is_even و is_odd بالعودية المتبادلة
:- dynamic is_even/1.
:- dynamic is_odd/1.

% الحالة الأساسية لـ is_even
is_even(0).

% is_even العودية: N زوجي إذا كان N-1 فرديًا
is_even(N) :-
    N > 0,
    N1 is N - 1,
    is_odd(N1).  % ← يستدعي is_odd

% الحالة الأساسية لـ is_odd
is_odd(1).

% is_odd العودية: N فردي إذا كان N-1 زوجيًا
is_odd(N) :-
    N > 1,
    N1 is N - 1,
    is_even(N1).  % ← يستدعي is_even

### الاختبار في Prolog

In [ ]:
% اختبار زوجي/فردي
is_even(0), writeln('✓ 0 is even').
is_even(4), writeln('✓ 4 is even').
is_odd(3), writeln('✓ 3 is odd').
is_odd(7), writeln('✓ 7 is odd').

### التحقق من العودية المتبادلة

In [ ]:
% بناء مخطط الاستدعاء والبحث عن SCCs
\+ \+ (
    use_module(unifyweaver(core/advanced/call_graph)),
    use_module(unifyweaver(core/advanced/scc_detection)),

    build_call_graph([is_even/1, is_odd/1], _Graph),
    format('Call graph: ~w~n', [_Graph]),

    find_sccs(_Graph, _SCCs),
    format('SCCs (mutual recursion groups): ~w~n', [_SCCs])
).

### التجميع إلى Bash

In [ ]:
% تجميع مجموعة العودية المتبادلة
\+ \+ (
    use_module(unifyweaver(core/advanced/mutual_recursion)),

    compile_mutual_recursion([is_even/1, is_odd/1], [], _BashCode),
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Main dispatch: route command line calls to functions"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/even_odd_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled is_even/is_odd to Bash with shared memoization')
).

### اختبار كود Bash المولد

In [ ]:
%%bash
source ../output/even_odd_demo.sh
echo "اختبار is_even و is_odd:"
is_even 0 >/dev/null && echo "✓ 0 زوجي"
is_even 4 >/dev/null && echo "✓ 4 زوجي"
is_odd 3 >/dev/null && echo "✓ 3 فردي"
is_odd 7 >/dev/null && echo "✓ 7 فردي"
is_even 5 >/dev/null 2>&1 || echo "✓ 5 ليس زوجيًا"

## مقارنة الأنماط

دعنا نقارن خصائص كل نمط:

| النمط | الاستدعاءات العودية | التحسين | التعقيد المكاني | الأفضل لـ |
|:--------|:----------------|:-------------|:-----------------|:---------|
| **عودية الذيل** | 1 (في موضع الذيل) | حلقة تكرارية | O(1) | المجمعات، المسح الخطي |
| **العودية الخطية** | 1 (في أي موضع) | طي (Fold) + حفظ في الذاكرة | O(n) لجدول الحفظ | فيبوناتشي، العاملي |
| **العودية الشجرية** | 2+ (على أجزاء الهيكل) | التفكيك الهيكلي | O(depth) لحجم المكدس | عمليات الأشجار والرسوم البيانية |
| **العودية المتبادلة** | 1+ (عبر المحددات) | حفظ مشترك في الذاكرة | O(n) للجدول المشترك | زوجي/فردي، التعريفات المتبادلة |

## ترتيب اكتشاف الأنماط

يحاول UnifyWeaver مطابقة الأنماط بهذا الترتيب:

1. **عودية الذيل** (الأكثر كفاءة)
2. **العودية الخطية** (ما لم تكن محظورة)
3. **العودية الشجرية** (الهيكلية)
4. **العودية المتبادلة** (اكتشاف المكونات شديدة الترابط SCC)
5. **العودية الأساسية** (البديل التلقائي)

يمكنك التأثير على عملية الاكتشاف باستخدام `forbid_linear_recursion/1`.

## تمرين: دورك الآن!

جرب تعريف هذه المحددات وتجميعها:

### 1. مجموع القائمة بعودية الذيل
```prolog
sum_list([], Acc, Acc).
sum_list([H|T], Acc, Sum) :-
    Acc1 is Acc + H,
    sum_list(T, Acc1, Sum).
```

### 2. فيبوناتشي بالعودية الخطية
```prolog
fib(0, 0).
fib(1, 1).
fib(N, F) :-
    N > 1,
    N1 is N - 1,
    N2 is N - 2,
    fib(N1, F1),
    fib(N2, F2),
    F is F1 + F2.
```

### 3. ارتفاع الشجرة
```prolog
tree_height([], 0).
tree_height([_, L, R], H) :-
    tree_height(L, HL),
    tree_height(R, HR),
    H is max(HL, HR) + 1.
```

In [ ]:
% كودك هنا!


## الملخص

في هذا الدفتر، تعلمت:

✅ أنماط العودية الأربعة الرئيسية في UnifyWeaver

✅ كيفية تعريف كل نمط في Prolog

✅ كيف يكتشف UnifyWeaver كل نمط ويحسنه

✅ خصائص الأداء لكل نمط

✅ متى تستخدم كل نمط

## الخطوات التالية

تابع إلى **دفتر الملاحظات 3: العرض المرئي لرسم الاستدعاءات** للتعرف على التحليل البرمجي المتقدم والعرض المرئي!